# Integration & Messaging

The moment you have more than one service, you need a way for them to talk that isn't "call me synchronously and hope I'm up." Azure has three distinct messaging products plus two integration services, and confusing them is the most common architecture mistake in the platform.

The three messengers — **Service Bus**, **Event Grid**, **Event Hubs** — look similar from a distance and are deeply different up close. The other two — **Logic Apps** and **API Management** — sit on top: Logic Apps as the no-code workflow runner, API Management as the API gateway. Together they cover the full integration story from per-message reliability up to public API exposure.

## Azure Service Bus — enterprise messaging

**Service Bus** is the classic enterprise message broker: AMQP, FIFO ordering, sessions, transactions, dead-letter queues, duplicate detection. It is what you reach for when *each message represents a unit of work that must be reliably processed exactly once, in order, with no loss*.

Two messaging primitives:

- **Queues** — point-to-point. One producer (or several) writes; one consumer (or a competing-consumer pool) reads. Each message goes to one receiver.
- **Topics & subscriptions** — pub-sub. Producers publish to a topic; multiple subscriptions filter and receive their own copy. Each subscription is, behind the scenes, a queue with filter rules.

Features that matter for real workloads:

- **Peek-lock receive** — the receiver locks the message for a configurable duration; completes it on success, abandons on failure. The default; gives you at-least-once.
- **Sessions** — group messages by `SessionId` so one consumer processes them in order. The Service-Bus way to do per-key FIFO.
- **Dead-letter queue (DLQ)** — every queue has a sibling DLQ; messages that fail delivery N times, or that you explicitly dead-letter, land there. Inspect, fix, re-enqueue.
- **Duplicate detection** — based on `MessageId` for a configurable window. Eliminates the "sender retried, now we have two" problem.
- **Scheduled enqueue** — drop a message that becomes visible at a future time.
- **Transactions** — atomic send/receive across multiple entities (queue + topic) within the same namespace.

Three tiers: **Basic** (queues only, no topics), **Standard** (full feature set, shared infra), **Premium** (dedicated capacity, predictable latency, geo-disaster recovery, VNet integration). Premium is the production tier for anything important.

AWS comparison: Service Bus ≈ SQS (queues) + SNS (topics) merged into one product with stronger ordering and transactional guarantees.

## Azure Event Grid — pub-sub for system events

**Event Grid** is the *push-based, lightweight* event router. It is built for reactive scenarios — "when a blob is uploaded, run this Function; when a resource is created, notify Logic Apps; when a custom event fires, fan it to four subscribers."

The architecture is push, not pull: subscribers register a webhook (Function, Logic App, Event Hub, Service Bus, custom HTTPS endpoint), and Event Grid POSTs events to it with retry and exponential backoff. Filtering happens at the subscription via subject, event type, or advanced property filters.

Three sources of events:

- **System topics** — Azure resources publish events automatically (Storage blob created/deleted, Key Vault secret expiring, Resource Group resource created). Subscribe and react.
- **Custom topics** — you publish your own events (`order.created`, `user.registered`).
- **Domains** — multi-tenant custom topic management; one publishing endpoint, per-tenant subscriptions.

Two event schemas: **Event Grid schema** (the original) and **CloudEvents 1.0** (the CNCF standard; recommended for portability).

Event Grid is for *event notifications*, not durable work queues. Each event is small (<= 64 KB), the goal is fast fan-out with at-least-once delivery, and failed deliveries dead-letter to a storage account if you configure it. If a message represents *work that must complete*, use Service Bus instead; if it represents *something happened, react if you want*, Event Grid is the fit.

AWS comparison: Event Grid ≈ EventBridge. The model — system events + custom events + filtered subscriptions — is virtually identical.

## Azure Event Hubs — big-data streaming

**Event Hubs** is the *high-throughput, append-only* event ingestion service. Think "millions of events per second from devices/clickstreams/logs into one durable stream that downstream processors read at their own pace."

The model is a partitioned log, deeply Kafka-like:

- **Partitions** — fixed at create (1–32 for Standard, more for Premium/Dedicated). Throughput scales with partitions. Each event lands on exactly one partition, picked by `partitionKey` for ordering or round-robin for spread.
- **Consumer groups** — each consumer group reads the full stream independently with its own offset. Five per Standard, 20 per Premium.
- **Throughput Units (TU)** / **Processing Units (PU)** — capacity blocks. 1 TU = 1 MB/s ingress, 2 MB/s egress. Premium uses PUs; Dedicated is single-tenant clusters.
- **Capture** — stream the events into ADLS Gen2 / Blob automatically. No consumer code needed for the "archive everything to the lake" pattern.
- **Kafka surface** — Event Hubs speaks the Kafka 1.0+ protocol on a different endpoint, so Kafka producers and consumers work unchanged. The most common reason to choose Event Hubs over Service Bus for a streaming workload.

Event Hubs retains events for a configurable window (1–90 days on Premium); consumers track their own offsets. There is no concept of "the event was processed." If you need acknowledgment-per-message semantics, Service Bus is the right tool, not Event Hubs.

AWS comparison: Event Hubs ≈ Kinesis Data Streams + MSK in one product. Capture ≈ Kinesis Firehose.

## Choosing between Service Bus, Event Grid, Event Hubs

The picking rule, summarised:

| Need | Service |
|------|---------|
| Reliable command/work queue with order and retries | **Service Bus** |
| Reactive notification of system or business events | **Event Grid** |
| High-volume telemetry / log / clickstream ingestion | **Event Hubs** |

Three useful questions:

1. **Is each message a unit of work that must complete exactly-or-at-least-once?** → Service Bus.
2. **Is each event a notification, and do you have several subscribers each doing something different?** → Event Grid.
3. **Is the volume millions per second and order/throughput matters more than per-message reliability?** → Event Hubs.

Real systems combine them. A typical pattern: clickstream events flow into **Event Hubs** for analytics; key events fan out via **Event Grid** to downstream services; one of those services is an order processor that pushes durable work onto a **Service Bus** queue for the warehouse. Each product earns its keep at its layer.

## Azure Logic Apps

**Logic Apps** is the no-code workflow runner — drag-and-drop triggers and actions wired through a graphical designer, with hundreds of pre-built **connectors** to Microsoft and SaaS systems (SAP, Salesforce, Office 365, Twilio, SQL, Outlook).

Two hosting models:

- **Consumption** — multi-tenant, per-action billing, no infrastructure to manage. Cold starts in seconds.
- **Standard** — single-tenant, runs on App Service plans (including in your VNet), local development with VS Code, multiple workflows per app. The production default for anything serious.

What Logic Apps does well: B2B integration with EDI/AS2/X12 via an **Integration Account**; SaaS-to-SaaS orchestration; "when an email arrives with a PDF attachment, OCR it, validate it, log to SharePoint" style workflows that would otherwise be a small developer project.

What it doesn't do well: high-throughput, low-latency message processing — that's Functions/Service Bus/Event Hubs territory. Logic Apps is for the *integration glue* between systems, not the data plane.

AWS comparison: Logic Apps ≈ Step Functions + EventBridge Pipes + AppFlow, with a much richer connector library and a graphical designer.

## Azure API Management

**API Management (APIM)** is the API gateway: a managed front-door for your HTTP APIs that handles auth, rate limiting, transformation, caching, mocking, and developer portal exposure.

Three customer-facing concepts:

- **APIs** — the operations you import (from OpenAPI, WSDL, App Service, Function App, Logic App, gRPC, GraphQL).
- **Products** — bundles of APIs you subscribe consumers to. Each subscription gets its own key; products carry per-product rate limits.
- **Policies** — XML snippets that run on every request: validate JWT, rewrite the URL, add CORS, cache for 60s, mock the response when the backend is down, return a different SLA tier to different products. The policy DSL is APIM's superpower.

Three SKU shapes:

- **Consumption** — serverless, per-call billing, no VNet, limited features. Good for hobby APIs.
- **Developer / Basic / Standard / Premium** — provisioned tiers; Premium adds VNet, multi-region deployment, self-hosted gateway, zone redundancy.
- **Standard v2 / Premium v2** — newer SKUs with better cost-to-feature, VNet support in Standard v2. Prefer v2 SKUs for new deployments.

**Self-hosted gateway** is a container you run in your data center (or in another cloud) that registers back to your APIM instance. Useful for hybrid scenarios where the backend is on-prem and you don't want traffic crossing the public internet to APIM and back.

The **developer portal** is an auto-generated, customisable site that publishes your APIs, lets consumers sign up, and gives them a subscription key. For external API products, it removes weeks of "build a portal" work.

AWS comparison: APIM ≈ API Gateway + parts of AWS Cognito (for subscription keys) + a developer portal AWS doesn't really provide.

In [ ]:
# A small messaging stack: Service Bus queue + Event Grid subscription + Function.

RG=rg-msg-demo
az group create -n $RG -l eastus

# 1. Service Bus namespace + queue with duplicate detection and a DLQ on max delivery 5.
az servicebus namespace create -g $RG -n sbns-foundations --sku Premium
az servicebus queue create -g $RG --namespace-name sbns-foundations -n orders \
  --enable-duplicate-detection true \
  --duplicate-detection-history-time-window PT10M \
  --max-delivery-count 5 \
  --enable-dead-lettering-on-message-expiration true

# 2. Event Grid custom topic + Storage subscription that dead-letters to a blob container.
az eventgrid topic create -g $RG -n egt-app --location eastus
az storage account create -g $RG -n stegdlq$RANDOM --sku Standard_LRS
az storage container create --account-name stegdlq$RANDOM --name eg-deadletter
az eventgrid event-subscription create \
  --name sub-to-function \
  --source-resource-id $(az eventgrid topic show -g $RG -n egt-app --query id -o tsv) \
  --endpoint-type AzureFunction \
  --endpoint /subscriptions/<sub>/resourceGroups/$RG/providers/Microsoft.Web/sites/func-orders/functions/HandleOrder \
  --deadletter-endpoint $(az storage account show -g $RG -n stegdlq$RANDOM --query id -o tsv)/blobServices/default/containers/eg-deadletter

## Putting it together

A typical event-driven Azure architecture, layered top to bottom:

1. **Public APIs** — fronted by **API Management** behind Front Door, with JWT validation, rate-limits per product, and request/response transformation policies.
2. **Synchronous calls** — APIM forwards to App Service / Container Apps / AKS / Functions, which serve the request directly.
3. **Reactive fan-out** — those services publish business events to **Event Grid** custom topics; downstream services subscribe to the events they care about. Notifications, audit logs, search index updates all fall out of this naturally.
4. **Durable work** — when an event triggers a unit of work that must complete (charge a card, ship an order, send a contract), it's enqueued on **Service Bus**. Workers run in Functions, Container Apps, or AKS, with DLQ and retries.
5. **Telemetry / streams** — clickstream, IoT, logs flow into **Event Hubs**, which both captures to ADLS Gen2 (analytics) and exposes the stream to Stream Analytics / Spark for processing.
6. **Workflows** — anything human-in-the-loop, B2B EDI, or SaaS-to-SaaS orchestration runs in **Logic Apps Standard**.

That stack covers most enterprise integration without anyone reaching for an ESB or rolling their own messaging. The discipline is keeping each tier to its proper job — Service Bus carries work, Event Grid notifies, Event Hubs streams, Logic Apps glues, APIM fronts. The four nouns are different on purpose.